In [ ]:
import pandas as pd
import numpy as np
import os
from dotenv import load_dotenv
load_dotenv()

import json
from tqdm import tqdm
import pandas as pd
data_path = "../../../data/external_data/fox8_23_dataset.ndjson"
with open(data_path, "r", encoding="utf-8") as f:
    data = [json.loads(line) for line in f]

fox8_23_dicts = []
for sample in tqdm(data):
    label = 0 if sample["label"] == "human" else 1
    user_id = sample["user_id"]
    for tweet in sample["user_tweets"]:
        fox8_23_dicts.append({
            "text": tweet["text"],
            "user_id": user_id,
            "label": label
        })
fox8_23 = pd.DataFrame(fox8_23_dicts)

In [ ]:
import os
import logging
from rouge import Rouge
from openai import OpenAI


client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
MODEL = "gpt-5-nano-2025-08-07"

class GECScore:
    def __init__(self, llm_model):
        """
        Args:
            llm_model (str): Name of the LLM model to use.
        """
        self.llm_model = llm_model
        self.rouge = Rouge()

    def _run_gec(self, text):
        """
        Runs the grammar error correction (GEC) model.
        """
        prompt = f"""
        You are a highly skilled grammar correction AI in multiple languages.
        You are provided with a text separated by <text></text> tags.
        Correct any grammatical, spelling, or punctuation errors in the text, doing only minimal changes necessary.
        Return ONLY the corrected text without any additional commentary.
        # Text to correct:
        <text>{text}</text>
        """
        return self.chat_with_openai(prompt, self.llm_model)
    
    def chat_with_openai(self, prompt, model=MODEL):
        try:
            completion = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "user", "content": prompt},
                ],
                reasoning_effort="minimal",
                seed=42
            )
            return completion.choices[0].message.content
        except Exception as e:
            logging.error(f"Error during LLM interaction: {e}")
            return None      

    def get_gecscore(self, text):
        """
        Processes a single item: runs GEC if needed, and computes ROUGE.
        """
        # 1. Generate gec_text if missing
        try:
            gec_text = self._run_gec(text)
        except Exception as e:
            logging.error(f"Error running GEC: {e}")

        # 2. Compute ROUGE score
        try:
            if gec_text:
                scores = self.rouge.get_scores(text, gec_text, avg=True)
                gec_score = scores['rouge-2']['f']
        except Exception as e:
            logging.warning(f"ROUGE computation failed: {e}")
            gec_score = None
        return gec_score

# Processing GEC scores for the validation set: 

In [ ]:
import concurrent.futures
import json
import pandas as pd
from tqdm import tqdm

def get_gecscore_for_row(row, i):
    text = row['text']
    try:
        score = gec_scorer.get_gecscore(text)
    except Exception as e:
        logging.error(f"Error getting GEC score for text: {e}")
        score = None   
    
    return {
        i: score
    }

# test with a row: 
i = 1
gec_scorer = GECScore(MODEL)
sample_row = fox8_23.iloc[i]
print(get_gecscore_for_row(sample_row, i))

# test with a row: 
i = -1
gec_scorer = GECScore(MODEL)
sample_row = fox8_23.iloc[i]
print(get_gecscore_for_row(sample_row, i))

In [ ]:
all_scores = {}
with concurrent.futures.ThreadPoolExecutor(max_workers=64) as executor:
    # Submit each row (as a Series) for processing
    futures = {executor.submit(get_gecscore_for_row, row, index): index for index, row in fox8_23.iterrows()}
    for future in tqdm(concurrent.futures.as_completed(futures), total=len(futures), desc="Gec Score Computation"):
        result = future.result()
        if result is not None:
            all_scores.update(result)

# Save results
import joblib
joblib.dump(all_scores, "gec_rouge2_scores_fox8.pkl")